# ASL Alphabet + Digits Fast Model

Train a separate fast model for:

- ASL alphabet: `A-Z`
- ASL digits: `0-9`

Total classes: 36.

This notebook uses `SEQ_LEN = 5` so recognition can be much faster than the word model that needs 20 frames. Image samples are converted into 5 repeated landmark frames.


In [ ]:
# CELL 1 - Install and download MediaPipe task models
!pip install mediapipe --quiet

import os
import urllib.request
import mediapipe as mp

print('MediaPipe', mp.__version__)

HAND_MODEL = '/kaggle/working/hand_landmarker.task'
POSE_MODEL = '/kaggle/working/pose_landmarker.task'

for path, url in [
    (HAND_MODEL, 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task'),
    (POSE_MODEL, 'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task'),
]:
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
        print(f'Downloaded {os.path.basename(path)}')

print('Ready')


In [ ]:
# CELL 2 - Imports
import json, os, cv2, glob, random, pickle, hashlib
import numpy as np
import tensorflow as tf
import mediapipe as mp
import matplotlib.pyplot as plt

from pathlib import Path
from collections import Counter
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

print('Imports OK')
print('TF:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:
# CELL 3 - Config
SEQ_LEN = 5
FEATURE_DIM = 225
BATCH_SIZE = 32
EPOCHS = 80
MAX_PER_CLASS = 800

SAVE_DIR = '/kaggle/working/'
LANDMARKS_DIR = '/kaggle/working/landmarks_alnum_fast/'
os.makedirs(LANDMARKS_DIR, exist_ok=True)

KAGGLE_INPUT = Path('/kaggle/input')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

LETTER_CLASSES = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
DIGIT_CLASSES = [str(i) for i in range(10)]
selected_words = LETTER_CLASSES + DIGIT_CLASSES

NUM_CLASSES = len(selected_words)
word_to_index = {w: i for i, w in enumerate(selected_words)}
index_to_word = {i: w for w, i in word_to_index.items()}

print(f'Classes ({NUM_CLASSES}):')
print(selected_words)
print(f'SEQ_LEN: {SEQ_LEN} | MAX_PER_CLASS: {MAX_PER_CLASS}')


In [ ]:
# CELL 4 - MediaPipe extractor
_hand_det = mp_vision.HandLandmarker.create_from_options(
    mp_vision.HandLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=HAND_MODEL),
        num_hands=2,
        min_hand_detection_confidence=0.3,
        min_hand_presence_confidence=0.3,
        min_tracking_confidence=0.3,
        running_mode=mp_vision.RunningMode.IMAGE,
    )
)
_pose_det = mp_vision.PoseLandmarker.create_from_options(
    mp_vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=POSE_MODEL),
        min_pose_detection_confidence=0.3,
        min_tracking_confidence=0.3,
        running_mode=mp_vision.RunningMode.IMAGE,
    )
)

def frame_to_landmarks(frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    hr = _hand_det.detect(mp_img)
    lh = np.zeros(63, dtype=np.float32)
    rh = np.zeros(63, dtype=np.float32)
    for i, handedness in enumerate(hr.handedness):
        if i >= len(hr.hand_landmarks):
            continue
        lms = np.array([[lm.x, lm.y, lm.z] for lm in hr.hand_landmarks[i]], dtype=np.float32).flatten()
        if handedness[0].category_name == 'Left':
            lh = lms
        else:
            rh = lms

    pr = _pose_det.detect(mp_img)
    pose = np.array([[lm.x, lm.y, lm.z] for lm in pr.pose_landmarks[0]], dtype=np.float32).flatten() \
           if pr.pose_landmarks else np.zeros(99, dtype=np.float32)

    return np.concatenate([lh, rh, pose]).astype(np.float32)

def extract_image_landmarks(image_path, seq_len=SEQ_LEN):
    image = cv2.imread(str(image_path))
    if image is None:
        return None
    try:
        features = frame_to_landmarks(image)
    except Exception:
        return None
    # Require at least one detected hand. Pose may be absent in cropped hand images.
    if not np.any(features[:126] != 0):
        return None
    return np.repeat(features.reshape(1, FEATURE_DIM), seq_len, axis=0).astype(np.float32)

print('Extractor ready')


In [ ]:
# CELL 5 - Scan alphabet/digit image datasets
def normalize_folder_label(name):
    clean = name.strip()
    upper = clean.upper()
    lower = clean.lower()
    if upper in LETTER_CLASSES:
        return upper
    if clean in DIGIT_CLASSES:
        return clean
    aliases = {
        'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
        'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9',
    }
    return aliases.get(lower)

def iter_labeled_images():
    for root, _, filenames in os.walk(KAGGLE_INPUT):
        label_name = normalize_folder_label(Path(root).name)
        if label_name is None or label_name not in word_to_index:
            continue
        for filename in filenames:
            path = Path(root) / filename
            if path.suffix.lower() in IMAGE_EXTENSIONS:
                yield path, label_name

def count_per_class():
    counts = Counter()
    for f in glob.glob(f'{LANDMARKS_DIR}*.npy'):
        try:
            label = int(Path(f).stem.rsplit('_', 1)[-1])
            counts[label] += 1
        except Exception:
            pass
    return counts

def save_sequence(seq, output_path):
    if seq is None or seq.shape != (SEQ_LEN, FEATURE_DIM):
        return False
    np.save(output_path, seq.astype(np.float32))
    return True

image_files = list(iter_labeled_images())
print(f'Discovered {len(image_files)} labeled images')

# Preview discovered class counts before extraction.
raw_counts = Counter(label for _, label in image_files)
for label in selected_words:
    print(f'{label:>2}: {raw_counts[label]} images')


In [ ]:
# CELL 6 - Extract image landmarks
counts = count_per_class()
saved = skipped = no_hand = 0

random.shuffle(image_files)
for image_path, label_name in image_files:
    label = word_to_index[label_name]
    if counts[label] >= MAX_PER_CLASS:
        skipped += 1
        continue

    digest = hashlib.md5(str(image_path).encode('utf-8')).hexdigest()[:12]
    npy_path = os.path.join(LANDMARKS_DIR, f'img_{label_name}_{digest}_{label}.npy')
    if os.path.exists(npy_path):
        counts[label] += 1
        saved += 1
        continue

    seq = extract_image_landmarks(image_path)
    if save_sequence(seq, npy_path):
        counts[label] += 1
        saved += 1
    else:
        no_hand += 1

    if (saved + no_hand) % 1000 == 0 and (saved + no_hand) > 0:
        print(f'processed={saved + no_hand} | saved={saved} | no_hand={no_hand}')

print(f'Images: {saved} saved | {skipped} over limit | {no_hand} no-hand/errors')
print(f'Total .npy: {len(glob.glob(LANDMARKS_DIR + "*.npy"))}')


In [ ]:
# CELL 7 - Check per-class counts
all_files = sorted(glob.glob(f'{LANDMARKS_DIR}*.npy'))
all_labels = []
for f in all_files:
    try:
        all_labels.append(int(Path(f).stem.rsplit('_', 1)[-1]))
    except Exception:
        all_labels.append(-1)

all_files = [f for f, l in zip(all_files, all_labels) if l in range(NUM_CLASSES)]
all_labels = [l for l in all_labels if l in range(NUM_CLASSES)]

counts = Counter(all_labels)
print(f'Total: {len(all_files)} samples | {len(counts)} / {NUM_CLASSES} classes')
print(f'\n{"Class":8} {"Count":>6} {"Status":>8}')
print('-' * 26)
for label in range(NUM_CLASSES):
    c = counts[label]
    status = 'OK' if c >= 80 else 'LOW' if c >= 30 else 'BAD'
    print(f'{index_to_word[label]:8} {c:>6} {status:>8}')

missing = [index_to_word[i] for i in range(NUM_CLASSES) if counts[i] == 0]
if missing:
    raise RuntimeError(f'Missing classes: {missing}')


In [ ]:
# CELL 8 - Normalize + balance + split
def normalize_sequence(seq):
    seq = seq.copy()
    for t in range(seq.shape[0]):
        lh = seq[t, 0:63].reshape(21, 3)
        if np.any(lh != 0):
            lh = lh - lh[0]
            s = np.max(np.linalg.norm(lh, axis=1)) + 1e-8
            seq[t, 0:63] = (lh / s).flatten()
        rh = seq[t, 63:126].reshape(21, 3)
        if np.any(rh != 0):
            rh = rh - rh[0]
            s = np.max(np.linalg.norm(rh, axis=1)) + 1e-8
            seq[t, 63:126] = (rh / s).flatten()
        pose = seq[t, 126:225].reshape(33, 3)
        if np.any(pose != 0):
            pose = pose - pose[0]
            sw = np.linalg.norm(pose[11] - pose[12]) + 1e-8
            seq[t, 126:225] = (pose / sw).flatten()
    return seq.astype(np.float32)

min_count = min(counts.values())
print(f'Balancing at {min_count} samples per class')

paired = list(zip(all_files, all_labels))
random.shuffle(paired)
class_seen = Counter()
bal_files, bal_labels = [], []
for f, l in paired:
    if class_seen[l] < min_count:
        bal_files.append(f)
        bal_labels.append(l)
        class_seen[l] += 1

X, y = [], []
for f, l in zip(bal_files, bal_labels):
    try:
        seq = np.load(f)
        if seq.shape == (SEQ_LEN, FEATURE_DIM):
            X.append(normalize_sequence(seq))
            y.append(l)
    except Exception:
        pass

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)
print(f'X: {X.shape} | y: {y.shape}')

train_X, val_X, train_y, val_y = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(train_X)} | Val: {len(val_X)}')


In [ ]:
# CELL 9 - Augmentation + tf.data pipeline
def augment_sequence(seq):
    seq = seq.copy()
    if np.random.rand() < 0.6:
        seq = seq + np.random.normal(0, 0.012, seq.shape).astype(np.float32)
    if np.random.rand() < 0.4:
        lh = seq[:, :63].copy()
        rh = seq[:, 63:126].copy()
        seq[:, :63] = rh
        seq[:, 63:126] = lh
        seq[:, 0::3] = -seq[:, 0::3]
    if np.random.rand() < 0.4:
        seq[:, :126] *= np.random.uniform(0.85, 1.15)
    if np.random.rand() < 0.4:
        shift = np.random.uniform(-0.08, 0.08, 3).astype(np.float32)
        seq[:, 0:63:3] += shift[0]
        seq[:, 1:63:3] += shift[1]
        seq[:, 63:126:3] += shift[0]
        seq[:, 64:126:3] += shift[1]
    if np.random.rand() < 0.2:
        drop_index = np.random.randint(0, SEQ_LEN)
        seq[drop_index] = np.zeros(FEATURE_DIM, dtype=np.float32)
    return seq.astype(np.float32)

def augment_tf(x, y):
    x = tf.numpy_function(augment_sequence, [x], tf.float32)
    x.set_shape((SEQ_LEN, FEATURE_DIM))
    return x, y

train_y_oh = tf.keras.utils.to_categorical(train_y, NUM_CLASSES)
val_y_oh = tf.keras.utils.to_categorical(val_y, NUM_CLASSES)

train_ds = tf.data.Dataset.from_tensor_slices((train_X, train_y_oh))
train_ds = train_ds.shuffle(len(train_X)).map(augment_tf).batch(BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((val_X, val_y_oh))
val_ds = val_ds.batch(BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)

STEPS_PER_EPOCH = max(1, len(train_X) // BATCH_SIZE)
VALIDATION_STEPS = max(1, len(val_X) // BATCH_SIZE)
print(f'Pipeline ready | steps:{STEPS_PER_EPOCH} | val_steps:{VALIDATION_STEPS}')


In [ ]:
# CELL 10 - Fast BiLSTM model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQ_LEN, FEATURE_DIM)),
    tf.keras.layers.LayerNormalization(),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(96, return_sequences=True)),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(48, return_sequences=False)),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(192, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.35),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()
print(f'Model ready - {NUM_CLASSES} classes | Random baseline: {100 / NUM_CLASSES:.1f}%')


In [ ]:
# CELL 11 - Train
history = model.fit(
    train_ds,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=VALIDATION_STEPS,
    epochs=EPOCHS,
    callbacks=[
        ModelCheckpoint(f'{SAVE_DIR}model_alnum.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=18, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1),
    ],
    verbose=1,
)
print('Training done')


In [ ]:
# CELL 12 - Save mapping + H5 + weights
best = tf.keras.models.load_model(f'{SAVE_DIR}model_alnum.keras')

mapping = {
    'word_to_index': word_to_index,
    'index_to_word': {str(k): v for k, v in index_to_word.items()},
    'selected_words': list(index_to_word.values()),
    'num_classes': NUM_CLASSES,
    'seq_len': SEQ_LEN,
    'feature_dim': FEATURE_DIM,
    'normalized': True,
    'architecture': 'asl_alphabet_digits_fast',
}
with open(f'{SAVE_DIR}mapping_alnum.json', 'w') as f:
    json.dump(mapping, f, indent=2)

with open(f'{SAVE_DIR}model_alnum_weights.pkl', 'wb') as f:
    pickle.dump(best.get_weights(), f)

best.save(f'{SAVE_DIR}model_alnum.h5')

print('Download from Kaggle output:')
for f in ['model_alnum.keras', 'model_alnum.h5', 'model_alnum_weights.pkl', 'mapping_alnum.json']:
    path = f'{SAVE_DIR}{f}'
    exists = os.path.exists(path)
    size = os.path.getsize(path) / 1024 / 1024 if exists else 0
    print(f"{'OK' if exists else 'NO'} {f:26} {size:.1f} MB")


In [ ]:
# CELL 13 - Accuracy check
preds_all = best.predict(val_X, verbose=0)
p = np.argmax(preds_all, axis=1)
t = val_y

correct = np.sum(p == t)
total = len(t)
top3_ok = sum(t[i] in np.argsort(preds_all[i])[-3:] for i in range(total))

print(f'Val Accuracy  : {correct / total * 100:.2f}% ({correct}/{total})')
print(f'Top-3 Accuracy: {top3_ok / total * 100:.2f}%')
print(f'Random baseline: {100 / NUM_CLASSES:.1f}%')
print('Sample predictions:')
for i in range(min(20, total)):
    pred = index_to_word[p[i]]
    real = index_to_word[t[i]]
    top3w = [index_to_word[j] for j in np.argsort(preds_all[i])[-3:][::-1]]
    print(f"  {'OK' if pred == real else 'NO'} Real:{real:4} Pred:{pred:4} Top3:{top3w}")


In [ ]:
# CELL 14 - Training curves and confusion matrix
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['accuracy'], label='Train', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val', linewidth=2)
ax1.axhline(y=1 / NUM_CLASSES, color='r', linestyle='--', label=f'Random ({100 / NUM_CLASSES:.1f}%)')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(alpha=0.3)
ax2.plot(history.history['loss'], label='Train', linewidth=2)
ax2.plot(history.history['val_loss'], label='Val', linewidth=2)
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(alpha=0.3)
plt.suptitle(f'Alphabet + Digits Fast Model ({NUM_CLASSES} classes)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}training_curve_alnum.png', dpi=150)
plt.show()

labels = list(range(NUM_CLASSES))
words = [index_to_word[i] for i in labels]
cm = confusion_matrix(t, p, labels=labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(cm_norm, cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
ax.set_xticks(labels)
ax.set_yticks(labels)
ax.set_xticklabels(words, rotation=90, ha='center', fontsize=8)
ax.set_yticklabels(words, fontsize=8)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix - Alnum Fast ({NUM_CLASSES} classes)')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}confusion_matrix_alnum.png', dpi=150)
plt.show()
